# Image Deblurring and Restoration using GAN for Number Plate Recognition

**How to use:** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`. Everything (data, training, evaluation, OCR test, saving the model) runs automatically. Expected time on a T4: roughly 15-25 minutes with the default settings.

**Pipeline**
1. Collect clean plate images (real plate crops from a public dataset, topped up with synthetic plates so training always works).
2. Degrade them on-the-fly (motion blur, Gaussian blur, down/up-scaling, noise, JPEG) to create *blurred -> sharp* pairs.
3. Train a conditional GAN: ResNet-based generator + PatchGAN discriminator, with L1 + VGG perceptual + LSGAN adversarial loss.
4. Evaluate with PSNR / SSIM and check the effect on plate **recognition** (EasyOCR) for blurred vs. restored images.

**Research basis** (please verify the exact citations on IEEE Xplore / CVF Open Access before putting them in your report)
- O. Kupyn et al., *DeblurGAN: Blind Motion Deblurring Using Conditional Adversarial Networks*, CVPR 2018.
- O. Kupyn et al., *DeblurGAN-v2: Deblurring (Orders-of-Magnitude) Faster and Better*, ICCV 2019.
- C. Ledig et al., *Photo-Realistic Single Image Super-Resolution Using a GAN (SRGAN)*, CVPR 2017.
- P. Isola et al., *Image-to-Image Translation with Conditional Adversarial Networks (pix2pix)*, CVPR 2017.
- X. Mao et al., *Least Squares Generative Adversarial Networks*, ICCV 2017.
- R. Laroca et al., *A Robust Real-Time Automatic License Plate Recognition Based on the YOLO Detector*, IJCNN 2018 (UFPR-ALPR dataset).

**What this notebook borrows:** DeblurGAN's residual generator with a global skip connection, SRGAN/DeblurGAN's perceptual (VGG) loss, pix2pix's PatchGAN discriminator, and LSGAN's stable adversarial loss.

In [ ]:
# ========== 1. Setup & configuration ==========
!pip -q install "datasets<4.0" scikit-image easyocr

import os, random, math, time, difflib, zipfile
import numpy as np, cv2, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image, ImageDraw, ImageFont
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim_fn

class CFG:
    H, W        = 64, 192          # plate size used for training
    EPOCHS      = 40
    BATCH       = 32
    LR          = 2e-4
    N_RES       = 9                # residual blocks in generator
    MIN_IMAGES  = 3000             # real plates are topped up with synthetic ones to reach this
    LAMBDA_L1   = 10.0
    LAMBDA_PERC = 1.0
    LAMBDA_ADV  = 0.01
    SEED        = 42
    OUT_DIR     = "/content/outputs"
    OWN_PLATES  = "/content/my_plates"   # optional: upload your own clean plate crops here

os.makedirs(CFG.OUT_DIR, exist_ok=True)
random.seed(CFG.SEED); np.random.seed(CFG.SEED); torch.manual_seed(CFG.SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "NO GPU - enable a GPU runtime!")

In [ ]:
# ========== 2. Get clean plate images ==========
# Priority: (a) your own folder, (b) public real-plate dataset from Hugging Face (no login needed),
# (c) synthetic plates to top up / as a guaranteed fallback.

def resize_plate(rgb):
    return cv2.resize(rgb, (CFG.W, CFG.H), interpolation=cv2.INTER_CUBIC)

real_plates = []

# (a) your own plates
if os.path.isdir(CFG.OWN_PLATES):
    for f in sorted(os.listdir(CFG.OWN_PLATES)):
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            im = cv2.imread(os.path.join(CFG.OWN_PLATES, f))
            if im is not None:
                real_plates.append(resize_plate(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)))
    print("Own plates loaded:", len(real_plates))

# (b) public dataset: full car photos + plate bounding boxes -> crop the plates
if len(real_plates) < 200:
    try:
        from datasets import load_dataset
        ds = load_dataset("keremberke/license-plate-object-detection", name="full", trust_remote_code=True)
        for split in ds.keys():
            for ex in ds[split]:
                img = ex["image"].convert("RGB")
                arr = np.array(img)
                boxes = ex["objects"]["bbox"]            # COCO format [x, y, w, h]
                for (x, y, w, h) in boxes:
                    x, y, w, h = int(x), int(y), int(w), int(h)
                    if w < 40 or h < 12:                  # skip tiny crops
                        continue
                    crop = arr[max(y, 0):y + h, max(x, 0):x + w]
                    if crop.size:
                        real_plates.append(resize_plate(crop))
        print("Real plate crops from Hugging Face:", len(real_plates))
    except Exception as e:
        print("Could not load the public dataset (", type(e).__name__, ") -> using synthetic plates only.")

# (c) synthetic plates
FONT_PATHS = ["/usr/share/fonts/truetype/dejavu/DejaVuSansMono-Bold.ttf",
              "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"]
def get_font(size):
    for p in FONT_PATHS:
        if os.path.exists(p):
            return ImageFont.truetype(p, size)
    return ImageFont.load_default()

LETTERS = "ABCDEFGHJKLMNPRSTUVWXYZ"
def random_plate_text(rng):
    style = rng.randint(3)
    if style == 0:   # Indian style  GJ05AB1234
        return f"{rng.choice(list('GMDKTU'))}{rng.choice(list('JHLAPN'))}{rng.randint(1,99):02d}" \
               f"{rng.choice(list(LETTERS))}{rng.choice(list(LETTERS))}{rng.randint(0,9999):04d}"
    if style == 1:   # ABC-1234
        return "".join(rng.choice(list(LETTERS), 3)) + str(rng.randint(1000, 9999))
    return "".join(rng.choice(list(LETTERS), 2)) + f"{rng.randint(10,99)}" + "".join(rng.choice(list(LETTERS), 3))

def synth_plate(rng):
    bg = [(255, 255, 255), (250, 250, 235), (255, 214, 0), (245, 245, 245)][rng.randint(4)]
    fg = (15, 15, 15) if bg != (255, 214, 0) else (20, 20, 20)
    scale = 4
    img = Image.new("RGB", (CFG.W * scale, CFG.H * scale), bg)
    d = ImageDraw.Draw(img)
    d.rectangle([4, 4, CFG.W * scale - 5, CFG.H * scale - 5], outline=fg, width=6)
    text = random_plate_text(rng)
    font = get_font(int(CFG.H * scale * 0.55))
    bbox = d.textbbox((0, 0), text, font=font)
    tw, th = bbox[2] - bbox[0], bbox[3] - bbox[1]
    if tw > CFG.W * scale * 0.9:
        font = get_font(int(CFG.H * scale * 0.55 * (CFG.W * scale * 0.9) / tw))
        bbox = d.textbbox((0, 0), text, font=font); tw, th = bbox[2] - bbox[0], bbox[3] - bbox[1]
    d.text(((CFG.W * scale - tw) // 2 - bbox[0], (CFG.H * scale - th) // 2 - bbox[1]), text, fill=fg, font=font)
    arr = np.array(img.resize((CFG.W, CFG.H), Image.LANCZOS))
    return arr, text

rng_s = np.random.RandomState(CFG.SEED)
all_plates = list(real_plates)
n_real = len(all_plates)
while len(all_plates) < CFG.MIN_IMAGES:
    all_plates.append(synth_plate(rng_s)[0])
print(f"Total clean images: {len(all_plates)}  (real: {n_real}, synthetic: {len(all_plates) - n_real})")

# train / val split
idx = np.random.RandomState(CFG.SEED).permutation(len(all_plates))
n_val = max(200, int(0.1 * len(all_plates)))
val_imgs   = [all_plates[i] for i in idx[:n_val]]
train_imgs = [all_plates[i] for i in idx[n_val:]]
print("Train:", len(train_imgs), "| Val:", len(val_imgs))

In [ ]:
# ========== 3. Degradation pipeline (creates the blurred input on-the-fly) ==========
def motion_kernel(size, angle):
    k = np.zeros((size, size), np.float32); k[size // 2, :] = 1
    M = cv2.getRotationMatrix2D((size / 2 - 0.5, size / 2 - 0.5), angle, 1)
    k = cv2.warpAffine(k, M, (size, size))
    return k / (k.sum() + 1e-8)

def degrade(img, rng):
    out = img.copy()
    r = rng.rand()
    if r < 0.45:
        out = cv2.filter2D(out, -1, motion_kernel(int(rng.choice([5, 7, 9, 11, 15, 19])), rng.uniform(0, 180)))
    elif r < 0.75:
        out = cv2.GaussianBlur(out, (0, 0), rng.uniform(0.8, 2.5))
    else:
        out = cv2.filter2D(out, -1, motion_kernel(int(rng.choice([5, 7, 9])), rng.uniform(0, 180)))
        out = cv2.GaussianBlur(out, (0, 0), rng.uniform(0.5, 1.5))
    if rng.rand() < 0.5:   # low resolution (super-resolution component)
        s = rng.uniform(0.4, 0.8)
        h, w = out.shape[:2]
        out = cv2.resize(cv2.resize(out, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA),
                         (w, h), interpolation=cv2.INTER_LINEAR)
    out = np.clip(out.astype(np.float32) + rng.randn(*out.shape) * rng.uniform(0, 6), 0, 255).astype(np.uint8)
    if rng.rand() < 0.5:   # JPEG artefacts
        _, enc = cv2.imencode(".jpg", out, [cv2.IMWRITE_JPEG_QUALITY, int(rng.randint(40, 95))])
        out = cv2.imdecode(enc, cv2.IMREAD_COLOR)
    return out

def to_tensor(a):  # uint8 HWC -> float CHW in [-1, 1]
    return torch.from_numpy(a).permute(2, 0, 1).float() / 127.5 - 1.0

class PlateDS(Dataset):
    def __init__(self, imgs, train=True):
        self.imgs, self.train = imgs, train
        if not train:   # fixed degradations for fair, repeatable validation
            self.blur = [degrade(im, np.random.RandomState(1000 + i)) for i, im in enumerate(imgs)]
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        sharp = self.imgs[i]
        if self.train:
            rng = np.random.RandomState((torch.initial_seed() * 1000003 + i + random.randint(0, 10**6)) % (2**32))
            blur = degrade(sharp, rng)
        else:
            blur = self.blur[i]
        return to_tensor(blur), to_tensor(sharp)

train_dl = DataLoader(PlateDS(train_imgs, True), batch_size=CFG.BATCH, shuffle=True, num_workers=2, drop_last=True, pin_memory=True)
val_ds   = PlateDS(val_imgs, False)
val_dl   = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0)

# quick look at the data
fig, ax = plt.subplots(2, 6, figsize=(18, 3.6))
for j in range(6):
    b, s = val_ds[j]
    ax[0, j].imshow(((b.permute(1, 2, 0) + 1) / 2).numpy()); ax[0, j].axis("off"); ax[0, j].set_title("blurred")
    ax[1, j].imshow(((s.permute(1, 2, 0) + 1) / 2).numpy()); ax[1, j].axis("off"); ax[1, j].set_title("sharp")
plt.tight_layout(); plt.show()

In [ ]:
# ========== 4. Models ==========
class ResBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.b = nn.Sequential(nn.ReflectionPad2d(1), nn.Conv2d(c, c, 3), nn.InstanceNorm2d(c), nn.ReLU(True),
                               nn.ReflectionPad2d(1), nn.Conv2d(c, c, 3), nn.InstanceNorm2d(c))
    def forward(self, x): return x + self.b(x)

class Generator(nn.Module):   # DeblurGAN-style ResNet generator with global skip connection
    def __init__(self, n_res=9, ngf=64):
        super().__init__()
        L = [nn.ReflectionPad2d(3), nn.Conv2d(3, ngf, 7), nn.InstanceNorm2d(ngf), nn.ReLU(True)]
        c = ngf
        for _ in range(2):
            L += [nn.Conv2d(c, c * 2, 3, 2, 1), nn.InstanceNorm2d(c * 2), nn.ReLU(True)]; c *= 2
        L += [ResBlock(c) for _ in range(n_res)]
        for _ in range(2):
            L += [nn.ConvTranspose2d(c, c // 2, 3, 2, 1, output_padding=1), nn.InstanceNorm2d(c // 2), nn.ReLU(True)]; c //= 2
        L += [nn.ReflectionPad2d(3), nn.Conv2d(c, 3, 7)]
        self.net = nn.Sequential(*L)
    def forward(self, x): return torch.clamp(self.net(x) + x, -1, 1)

class Discriminator(nn.Module):   # PatchGAN
    def __init__(self, ndf=64):
        super().__init__()
        def blk(i, o, s, norm=True):
            return [nn.Conv2d(i, o, 4, s, 1)] + ([nn.InstanceNorm2d(o)] if norm else []) + [nn.LeakyReLU(0.2, True)]
        self.net = nn.Sequential(*blk(3, ndf, 2, False), *blk(ndf, ndf * 2, 2), *blk(ndf * 2, ndf * 4, 2),
                                 *blk(ndf * 4, ndf * 8, 1), nn.Conv2d(ndf * 8, 1, 4, 1, 1))
    def forward(self, x): return self.net(x)

class VGGPerceptual(nn.Module):
    def __init__(self):
        super().__init__()
        from torchvision.models import vgg19, VGG19_Weights
        self.f = vgg19(weights=VGG19_Weights.IMAGENET1K_V1).features[:23].eval()
        for p in self.f.parameters(): p.requires_grad = False
        self.register_buffer("m", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("s", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
    def forward(self, x):
        x = ((x + 1) / 2 - self.m) / self.s
        return self.f(x)

G, D, vgg = Generator(CFG.N_RES).to(device), Discriminator().to(device), VGGPerceptual().to(device)
print(f"Generator params: {sum(p.numel() for p in G.parameters())/1e6:.2f}M | Discriminator params: {sum(p.numel() for p in D.parameters())/1e6:.2f}M")

In [ ]:
# ========== 5. Training ==========
optG = torch.optim.Adam(G.parameters(), lr=CFG.LR, betas=(0.5, 0.999))
optD = torch.optim.Adam(D.parameters(), lr=CFG.LR, betas=(0.5, 0.999))
decay = lambda e: 1.0 if e < CFG.EPOCHS // 2 else 1.0 - (e - CFG.EPOCHS // 2) / (CFG.EPOCHS - CFG.EPOCHS // 2 + 1)
schG, schD = torch.optim.lr_scheduler.LambdaLR(optG, decay), torch.optim.lr_scheduler.LambdaLR(optD, decay)

def psnr_batch(a, b):   # tensors in [-1, 1]
    mse = (((a + 1) / 2 - (b + 1) / 2) ** 2).flatten(1).mean(1).clamp_min(1e-10)
    return (10 * torch.log10(1.0 / mse))

@torch.no_grad()
def validate(model):
    model.eval(); ps_in, ps_out = [], []
    for b, s in val_dl:
        b, s = b.to(device), s.to(device)
        ps_in.append(psnr_batch(b, s).cpu()); ps_out.append(psnr_batch(model(b), s).cpu())
    model.train()
    return torch.cat(ps_in).mean().item(), torch.cat(ps_out).mean().item()

hist = {"G": [], "D": [], "L1": [], "perc": [], "psnr": []}
best_psnr, t0 = -1, time.time()
base_psnr, _ = validate(G)
print(f"PSNR of blurred input vs sharp (baseline): {base_psnr:.2f} dB\n")

for epoch in range(CFG.EPOCHS):
    eG = eD = eL1 = eP = 0.0; n = 0
    for blur, sharp in train_dl:
        blur, sharp = blur.to(device, non_blocking=True), sharp.to(device, non_blocking=True)
        fake = G(blur)

        # --- Discriminator (LSGAN) ---
        optD.zero_grad()
        lD = 0.5 * (F.mse_loss(D(sharp), torch.ones_like(D(sharp))) + F.mse_loss(D(fake.detach()), torch.zeros_like(D(fake.detach()))))
        lD.backward(); optD.step()

        # --- Generator: L1 + perceptual + adversarial ---
        optG.zero_grad()
        pred = D(fake)
        l_adv  = F.mse_loss(pred, torch.ones_like(pred))
        l_l1   = F.l1_loss(fake, sharp)
        l_perc = F.mse_loss(vgg(fake), vgg(sharp))
        lG = CFG.LAMBDA_L1 * l_l1 + CFG.LAMBDA_PERC * l_perc + CFG.LAMBDA_ADV * l_adv
        lG.backward(); optG.step()

        eG += lG.item(); eD += lD.item(); eL1 += l_l1.item(); eP += l_perc.item(); n += 1
    schG.step(); schD.step()
    _, p_out = validate(G)
    for k, v in zip(["G", "D", "L1", "perc", "psnr"], [eG / n, eD / n, eL1 / n, eP / n, p_out]): hist[k].append(v)
    if p_out > best_psnr:
        best_psnr = p_out; torch.save(G.state_dict(), f"{CFG.OUT_DIR}/generator_best.pth")
    print(f"Epoch {epoch+1:02d}/{CFG.EPOCHS} | G {eG/n:.3f} | D {eD/n:.3f} | L1 {eL1/n:.4f} | val PSNR {p_out:.2f} dB "
          f"| best {best_psnr:.2f} | {(time.time()-t0)/60:.1f} min")

torch.save(G.state_dict(), f"{CFG.OUT_DIR}/generator_last.pth")
torch.save(D.state_dict(), f"{CFG.OUT_DIR}/discriminator_last.pth")
print("\nTraining finished.")

In [ ]:
# ========== 6. Results: curves, PSNR / SSIM, visual comparison ==========
G.load_state_dict(torch.load(f"{CFG.OUT_DIR}/generator_best.pth", map_location=device)); G.eval()

fig, ax = plt.subplots(1, 3, figsize=(16, 3.5))
ax[0].plot(hist["G"], label="G"); ax[0].plot(hist["D"], label="D"); ax[0].set_title("GAN losses"); ax[0].legend()
ax[1].plot(hist["L1"]); ax[1].set_title("Generator L1 loss")
ax[2].plot(hist["psnr"]); ax[2].axhline(base_psnr, color="r", ls="--", label="blurred input"); ax[2].set_title("Validation PSNR (dB)"); ax[2].legend()
plt.tight_layout(); plt.savefig(f"{CFG.OUT_DIR}/training_curves.png", dpi=150); plt.show()

def tensor_to_np(t): return ((t.clamp(-1, 1).permute(1, 2, 0).cpu().numpy() + 1) / 2 * 255).astype(np.uint8)

restored_all, blur_all, sharp_all = [], [], []
with torch.no_grad():
    for b, s in val_dl:
        r = G(b.to(device)).cpu()
        for i in range(len(b)):
            blur_all.append(tensor_to_np(b[i])); sharp_all.append(tensor_to_np(s[i])); restored_all.append(tensor_to_np(r[i]))

def mean_ssim(A, B):
    return float(np.mean([ssim_fn(a, b, channel_axis=2, data_range=255) for a, b in zip(A, B)]))
def mean_psnr(A, B):
    return float(np.mean([cv2.PSNR(a, b) for a, b in zip(A, B)]))

print(f"{'':22s}{'PSNR (dB)':>12s}{'SSIM':>10s}")
print(f"{'Blurred input':22s}{mean_psnr(blur_all, sharp_all):12.2f}{mean_ssim(blur_all, sharp_all):10.4f}")
print(f"{'GAN restored':22s}{mean_psnr(restored_all, sharp_all):12.2f}{mean_ssim(restored_all, sharp_all):10.4f}")

n_show = 8
fig, ax = plt.subplots(3, n_show, figsize=(2.6 * n_show, 3.4))
for j in range(n_show):
    for r, (arr, name) in enumerate([(blur_all, "Blurred"), (restored_all, "Restored"), (sharp_all, "Ground truth")]):
        ax[r, j].imshow(arr[j]); ax[r, j].axis("off")
        if j == 0: ax[r, j].set_title(name, loc="left", fontsize=10)
plt.tight_layout(); plt.savefig(f"{CFG.OUT_DIR}/comparison.png", dpi=150); plt.show()

In [ ]:
# ========== 7. Does restoration help plate RECOGNITION? (EasyOCR) ==========
# Reference text = OCR output on the sharp image (so it also works for real plates without labels).
try:
    import easyocr
    reader = easyocr.Reader(["en"], gpu=(device.type == "cuda"), verbose=False)

    def read(img):
        big = cv2.resize(img, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)
        res = reader.readtext(big, allowlist="ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789", detail=0)
        return "".join(res).upper()

    N = min(200, len(sharp_all))
    ref  = [read(x) for x in sharp_all[:N]]
    keep = [i for i, t in enumerate(ref) if len(t) >= 4]           # only plates the OCR can read when sharp
    def score(imgs):
        exact, sim = [], []
        for i in keep:
            t = read(imgs[i]); exact.append(t == ref[i]); sim.append(difflib.SequenceMatcher(None, t, ref[i]).ratio())
        return 100 * np.mean(exact), 100 * np.mean(sim)

    eb, sb = score(blur_all); er, sr = score(restored_all)
    print(f"Evaluated on {len(keep)} plates (reference = OCR of sharp image)")
    print(f"{'':16s}{'Exact match %':>16s}{'Char similarity %':>20s}")
    print(f"{'Blurred':16s}{eb:16.1f}{sb:20.1f}")
    print(f"{'GAN restored':16s}{er:16.1f}{sr:20.1f}")
except Exception as e:
    print("OCR evaluation skipped:", type(e).__name__, e)

In [ ]:
# ========== 8. Save model & provide a reusable inference function ==========
def restore_image(path, model=G):
    # Restore ANY plate image (cropped plate). It is resized to 64x192, restored, and returned as RGB uint8.
    im = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    x = to_tensor(cv2.resize(im, (CFG.W, CFG.H), interpolation=cv2.INTER_CUBIC)).unsqueeze(0).to(device)
    with torch.no_grad():
        return tensor_to_np(model(x)[0].cpu())
# Example:  from google.colab import files; up = files.upload(); out = restore_image(list(up)[0]); plt.imshow(out)

zip_path = "/content/deblur_gan_plate_outputs.zip"
with zipfile.ZipFile(zip_path, "w") as z:
    for f in os.listdir(CFG.OUT_DIR): z.write(os.path.join(CFG.OUT_DIR, f), f)
print("Saved:", os.listdir(CFG.OUT_DIR))
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Download manually from:", zip_path)

## Notes for your report
- **Method:** conditional GAN for blind deblurring/restoration; loss = 10·L1 + 1·VGG-perceptual + 0.01·LSGAN (adversarial).
- **Data caveat:** many real plate crops are small, so the "sharp" targets are upscaled to 64x192 and are themselves slightly soft. Synthetic plates are perfectly sharp, which makes metrics look better on them. State the real/synthetic mix printed in section 2.
- **Evaluation caveat:** the OCR test uses EasyOCR on the sharp image as reference, so it measures *recovery of readability*, not absolute OCR accuracy.
- **To improve:** upload more real plates to `/content/my_plates`, raise `EPOCHS`, or add a plate-recognition-aware loss (OCR/CTC loss) as an extension.